In [94]:
import torch
print(torch.__version__)


2.14.1+cpu


In [95]:
text = open("input.txt", "r", encoding="utf-8").read()
print(len(text))
print(text[:1000])

1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for re

In [96]:
chars=sorted(set(text))

In [97]:
vocab_size=len(chars)

In [98]:
vocab_size

65

In [99]:
''.join(chars)

"\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"

In [100]:
stoi={ch:i for i,ch in enumerate(chars)}
itos={i:ch for i,ch in enumerate(chars)}

In [101]:
def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return ''.join([itos[i] for i in l])

In [102]:
s = "Hello World"
round_trip = decode(encode(s))
print(f"encode('{s}') = {encode(s)}")
print(f"decode(encode('{s}')) = {round_trip}'")
print(f"[{'✓' if round_trip == s else '✗'}] round-trip")

encode('Hello World') = [20, 43, 50, 50, 53, 1, 35, 53, 56, 50, 42]
decode(encode('Hello World')) = Hello World'
[✓] round-trip


In [103]:
data=torch.tensor(encode(text),dtype=torch.long)

In [104]:
data.shape

torch.Size([1115394])

In [105]:
n=int(0.9*len(data))
train_data=data[:n]
val_data=data[n:]

In [106]:
print(f"train has {len(train_data)} tokens")
print(f"val has {len(val_data)} tokens")

train has 1003854 tokens
val has 111540 tokens


In [107]:
print(data[:100])

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [108]:
def get_batch(split,block_size,batch_size):
    data=train_data if split=="train" else val_data
    start=torch.randint(0,len(data)-block_size,(batch_size,))
    xb=torch.stack([data[i:i+block_size] for i in start])
    yb=torch.stack([data[i+1:i+block_size+1] for i in start])
    return xb,yb

In [109]:
xb, yb = get_batch('train', block_size=8, batch_size=4)
print(f"xb.shape = {xb.shape}")   # 预期 torch.Size([4, 8])
print(f"yb.shape = {yb.shape}")   # 预期 torch.Size([4, 8])
print()
print("xb:")
print(xb)
print("yb:")
print(yb)

xb.shape = torch.Size([4, 8])
yb.shape = torch.Size([4, 8])

xb:
tensor([[46, 43,  1, 60, 53, 47, 41, 43],
        [53, 51, 43,  6,  1, 41, 47, 60],
        [53, 59,  1, 58, 46, 43, 56, 43],
        [58, 39, 58, 47, 53, 52,  8,  1]])
yb:
tensor([[43,  1, 60, 53, 47, 41, 43,  1],
        [51, 43,  6,  1, 41, 47, 60, 47],
        [59,  1, 58, 46, 43, 56, 43,  1],
        [39, 58, 47, 53, 52,  8,  1, 15]])


In [110]:
import torch.nn as nn
import torch.nn.functional as F


In [111]:
class BigramLM(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.token_embedding_table=nn.Embedding(vocab_size,vocab_size)

    def forward(self,idx,targets=None):
        logits=self.token_embedding_table(idx)

        if targets is None:
            loss=None
        else:
            B,T,C=logits.shape
            logits_flat=logits.view(B*T,C)
            targets_flat=targets.view(B*T)
            loss=F.cross_entropy(logits_flat,targets_flat)

        return logits,loss
    
    def generate(self,idx,max_new_tokens):
        for _ in range(max_new_tokens):
            logits,_=self.forward(idx)
            logits=logits[:,-1,:]
            prob=F.softmax(logits,dim=-1)
            next_token=torch.multinomial(prob,num_samples=1)
            idx=torch.cat((idx,next_token),dim=1)
        return idx

In [112]:
m = BigramLM(vocab_size)

# 测试 1: 没有 target 的 forward
xb, yb = get_batch('train', block_size=8, batch_size=4)
logits, loss = m(xb)
print(f"[{'✓' if tuple(logits.shape) == (4, 8, 65) else '✗'}] logits shape = {logits.shape}")
print(f"[{'✓' if loss is None else '✗'}] loss is None when targets=None")

# 测试 2: 带 target 的 forward
logits, loss = m(xb, yb)
print(f"[{'✓' if tuple(logits.shape) == (4, 8, 65) else '✗'}] logits shape = {logits.shape}")
print(f"[{'✓' if loss.ndim == 0 else '✗'}] loss is a scalar: {loss.item():.4f}")

# 测试 3: 初始 loss 应该接近 ln(vocab_size)
import math
expected_initial_loss = math.log(vocab_size)
print(f"  Initial loss {loss.item():.4f}  vs  ln(65) = {expected_initial_loss:.4f}")
print(f"[{'✓' if abs(loss.item() - expected_initial_loss) < 0.5 else '✗'}] 接近 ln(V)（证明初始化合理）")


[✓] logits shape = torch.Size([4, 8, 65])
[✓] loss is None when targets=None
[✓] logits shape = torch.Size([4, 8, 65])
[✓] loss is a scalar: 4.8506
  Initial loss 4.8506  vs  ln(65) = 4.1744
[✗] 接近 ln(V)（证明初始化合理）


In [113]:
optimizer=torch.optim.AdamW(m.parameters(),lr=1e-3)

In [114]:
batch_size=32
block_size=8
num_steps=3000

for step in range(num_steps):
    xb,yb=get_batch('train',block_size,batch_size)
    logits,loss=m(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step%300==0:
        print(f"step {step}: train loss {loss.item():.4f}")

step 0: train loss 4.7287
step 300: train loss 4.4691
step 600: train loss 4.1382
step 900: train loss 3.9114
step 1200: train loss 3.6247
step 1500: train loss 3.4631
step 1800: train loss 3.1983
step 2100: train loss 3.1652
step 2400: train loss 3.0201
step 2700: train loss 2.7732


In [115]:
# 新 cell —— 训练后验证
final_loss = loss.item()
print(f"\n=== Training done ===")
print(f"[{'✓' if final_loss < 3.0 else '✗'}] final loss {final_loss:.4f} < 3.0 (下降了)")
print(f"[{'✓' if final_loss > 2.0 else '✗'}] final loss {final_loss:.4f} > 2.0 (bigram 的理论下限约 2.4)")



=== Training done ===
[✓] final loss 2.7776 < 3.0 (下降了)
[✓] final loss 2.7776 > 2.0 (bigram 的理论下限约 2.4)


In [117]:
# 验证生成功能
start=torch.zeros((1,1),dtype=torch.long)
out = m.generate(start, max_new_tokens=100)
print(f"[{'✓' if tuple(out.shape) == (1, 101) else '✗'}] shape (1, 101), 得 {tuple(out.shape)}")

generated_text = decode(out[0].tolist())
print(f"[{'✓' if len(generated_text) == 101 else '✗'}] decoded 长度 101, 得 {len(generated_text)}")

# 应该大部分字符都在 vocab 内
all_in_vocab = all(c in stoi for c in generated_text)
print(f"[{'✓' if all_in_vocab else '✗'}] 生成的所有字符都在 vocab 内")

print(f"\n--- 生成样本 ---\n{generated_text}")

[✓] shape (1, 101), 得 (1, 101)
[✓] decoded 长度 101, 得 101
[✓] 生成的所有字符都在 vocab 内

--- 生成样本 ---

sixJz$OMOTN at ha 
TEG&$&Fr, bret cr dwDeNU-iD-s wojg ubs b d'CO:


FZO:?DIpSTh$$bofodlaOFEDiGmer mQ
